# Generar ganancias de ensambles

Conversión del script `generar_ganancias_ensambles.R` a un notebook para facilitar su ejecución paso a paso.

In [ ]:
#!/usr/bin/env Rscript


if (!require("data.table")) install.packages("data.table", repos = "http://cran.us.r-project.org")
library(data.table)

if (!require("openxlsx")) install.packages("openxlsx", repos = "http://cran.us.r-project.org")
library(openxlsx)

if (!require("stringr")) install.packages("stringr", repos = "http://cran.us.r-project.org")
library(stringr)

if (!require("lightgbm")) install.packages("lightgbm", repos = "http://cran.us.r-project.org")
library(lightgbm)




In [ ]:
library(data.table)

# --- CONFIGURACIÓN ---
# Directorio donde están los prediccion_ensamble*.txt
setwd("/content/buckets/b1/exp/ensemble_final")   # CAMBIAR si hace falta

# Patrón de archivos a ensamblar
patron_archivos <- "^prediccion_ensamble.*\\.txt$"

# Cortes para Kaggle
cortes_envios <- c(
  8500L, 9000L, 9500L, 10000L,
  10500L, 11000L, 11500L, 12000L,
  12500L, 13000L
)

# Nombre del experimento para los archivos de salida
experimento <- if (exists("PARAM") && !is.null(PARAM$experimento)) {
  PARAM$experimento
} else {
  "ensamble_final"
}

# --- LISTAR ARCHIVOS ---
archivos <- list.files(pattern = patron_archivos)

if (length(archivos) == 0L) {
  stop("No encontré archivos que cumplan el patrón: ", patron_archivos)
}

cat("Voy a ensamblar estos archivos:\n")
print(archivos)

# --- LEER Y VALIDAR ---
lista <- list()

for (i in seq_along(archivos)) {
  dt <- fread(archivos[i])

  # Detecto la columna de probabilidad
  score_col <- NULL
  if ("prob_ensamble" %in% names(dt)) {
    score_col <- "prob_ensamble"
  } else if ("prob" %in% names(dt)) {
    score_col <- "prob"
  } else {
    stop("En el archivo ", archivos[i],
         " no encuentro ni 'prob_ensamble' ni 'prob'")
  }

  # Me quedo solo con numero_de_cliente + prob
  dt <- dt[, .(numero_de_cliente, prob = get(score_col))]

  setkey(dt, numero_de_cliente)

  lista[[i]] <- dt
}

# Validaciones: mismo nrow y mismos clientes
base_clientes <- lista[[1]][, numero_de_cliente]
n_base <- length(base_clientes)

for (i in seq_along(lista)) {
  dt <- lista[[i]]

  # 1) misma cantidad de filas
  if (nrow(dt) != n_base) {
    stop("El archivo ", archivos[i], " tiene ",
         nrow(dt), " filas, pero el primero tiene ", n_base)
  }

  # 2) mismos clientes y en el mismo orden
  if (!identical(dt[, numero_de_cliente], base_clientes)) {
    stop("Los clientes del archivo ", archivos[i],
         " no coinciden (en orden) con el primero.")
  }
}

cat("Validación OK: todos los archivos tienen mismos clientes y misma cantidad de filas.\n")

# --- ENSAMBLE FINAL (PROMEDIO DE PROB POR CLIENTE) ---

dt_largo <- rbindlist(
  lapply(seq_along(lista), function(i) {
    lista[[i]][, modelo := i]
  })
)

ensamble_final <- dt_largo[, .(
  prob_ensamble = mean(prob)
), by = numero_de_cliente]

# Guardo el ensamble de probabilidades
fwrite(
  ensamble_final,
  file = "prediccion_ensamble_final.txt",
  sep  = "\t"
)

cat("Grabé prediccion_ensamble_final.txt con probs ensambladas.\n")

# --- GENERAR ARCHIVOS PARA KAGGLE Y ARCHIVOS SOLO IDs ---

# Ordeno por probabilidad descendente
setorder(ensamble_final, -prob_ensamble)

dir.create("kaggle_ensamble", showWarnings = FALSE)

for (corte in cortes_envios) {
  tb_envio <- copy(ensamble_final)

  tb_envio[, Predicted := 0L]
  tb_envio[1:corte, Predicted := 1L]

  # Archivo Kaggle estándar: numero_de_cliente, Predicted (con header)
  archivo_kaggle <- paste0(
    "kaggle_ensamble/KA_", experimento, "_", corte, ".csv"
  )

  fwrite(
    tb_envio[, .(numero_de_cliente, Predicted)],
    file      = archivo_kaggle,
    sep       = ",",
    col.names = TRUE
  )

  # Archivo SOLO IDs de los que quedaron en 1, sin header
  archivo_ids <- paste0(
    "kaggle_ensamble/KA_", experimento, "_", corte, "_ids.csv"
  )

  fwrite(
    tb_envio[Predicted == 1L, .(numero_de_cliente)],
    file      = archivo_ids,
    sep       = ",",
    col.names = FALSE
  )

  cat("Generado:\n  ", archivo_kaggle, "\n  ", archivo_ids, "\n")
}
